# Prerequisites 

In [4]:
%cd /dss/dsshome1/03/ge87wod2/morphological-inflection/ByT5/
#import sys
#print(sys.version)
#!python3 --version

/dss/dsshome1/03/ge87wod2/morphological-inflection/ByT5


/dss/dsshome1/03/ge87wod2/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
!pip uninstall transformers -y
!pip install transformers==4.36.2  datasets==4.3.0 accelerate==0.27.2
!pip install torch==2.9.1
!pip install ipywidgets==8.1.7
!pip install python-dotenv==1.2.1 pickleshare==0.7.5

  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-0.27.2-py3-none-any.whl.metadata (18 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached regex-2026.4.4-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.15.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached pyarrow-23.0.1-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.6.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.16-py310-none-any.whl.metadata (7.2 kB)
  Using cached torch-2.11.0-cp310-cp

  Using cached torch-2.9.1-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cusolver_cu12-11.7.3.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cusparse_cu12-12.5.8.93-py3-n

Using cached ipywidgets-8.1.7-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached pickleshare-0.7.5-py2.py3-none-any.whl.metadata (1.5 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
Using cached pickleshare-0.7.5-py2.py3-none-any.whl (6.9 kB)


In [1]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env",override=True)

HF_TOKEN = os.getenv("HF_TOKEN")

#print("HF_TOKEN: ", HF_TOKEN)

 # source: https://huggingface.co/docs/transformers/v4.28.1/tasks/summarization
from huggingface_hub import login
login(HF_TOKEN)

ModuleNotFoundError: No module named 'dotenv'

# Continued Pre-training

In [9]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset
import random
# get tokenizer and sentinal tokens for masking
tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
extra_id_0 = tokenizer.convert_tokens_to_ids("<extra_id_0>")
extra_id_1 = tokenizer.convert_tokens_to_ids("<extra_id_1>")
print(extra_id_0, extra_id_1)

# Load pre-trained model
model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, return_tensors="pt")
random.seed(10)
# Masking by processing the data line by line
def preprocess_function(examples): # receives unmasked sentence as input
    # get input sentence
    input_ids_batch = []
    attention_mask_batch = []
    labels_batch = []
    
    inputs_batch = examples["input"]
    for inputs in inputs_batch:
        inputs = inputs.strip()
        model_inputs = tokenizer(inputs)
        input_ids = model_inputs.input_ids
        attention_mask = model_inputs.attention_mask
        # print(input_ids, attention_mask, len(input_ids),len(attention_mask))
        
        # mask 15% of the bytes randomly in the tokenized sentence leaving out bordering bytes.
        length = round(len(input_ids) * 0.25) # mask 20% of bytes (T5 and mT5 do that for tokens)
        max_start_index = len(input_ids) - length -1  # 2 bytes + eos byte-> 3 - 1x eos - 1= 1 -> index 0 or 1 possible, eos does not count to bytes
        start_index = random.randrange(1, max_start_index -1) # leave out first bytes and last byte "full stop: 49"
        exclusive_end_index = start_index + length
        masked_input_ids = (  input_ids[:start_index] + [extra_id_0] + input_ids[exclusive_end_index :]  )
        masked_labels_ids = ([extra_id_0]+ input_ids[start_index: exclusive_end_index] +[extra_id_1])
        
        # update attention mask and labels and add to batch
        attention_mask = [1] * len(masked_input_ids)
        input_ids_batch.append(masked_input_ids)
        attention_mask_batch.append(attention_mask)
        labels_batch.append(masked_labels_ids)
    return {
        "input_ids": input_ids_batch,
        "attention_mask": attention_mask_batch,
        "labels": labels_batch
    }

# load dataset
lang = "csb-pol-sent"
train_path = f"/dss/dsshome1/03/ge87wod2/morphological-inflection/preprocessing/preprocessing_to_json_or_tsv/data/{lang}_trn.tsv"
eval_path = f"/dss/dsshome1/03/ge87wod2/morphological-inflection/preprocessing/preprocessing_to_json_or_tsv/data/{lang}_dev.tsv"
dataset = load_dataset("csv", delimiter="\t",column_names=["input"],data_files={"train": train_path,"validation": eval_path},quoting=3) # disable quote parsing
tokenized_dataset = dataset.map(preprocess_function, batched=True)
print(tokenized_dataset["train"]["input_ids"])
print(tokenized_dataset)

#training_args = Seq2SeqTrainingArguments(
#    output_dir="./byt5_small/csb-pol-byte-25perc-sent",
#    evaluation_strategy="epoch",
#    learning_rate=5e-5,
#    per_device_train_batch_size=16,
#    per_device_eval_batch_size=16,
#    weight_decay=0.01,
#    save_total_limit=3,
#    num_train_epochs=4,
#    predict_with_generate=True,
#    # save_strategy="epoch",
#    fp16=False,# did not work with True #change to bf16=True for XPU 
#    #bf16 = True, # performance was best without setting fp16 or bf16 to true
#    push_to_hub=True,
#    warmup_steps = 500,
#    # load_best_model_at_end=True, # otherwise not model with minimum loss during training, like https://huggingface.co/docs/transformers/tasks/sequence_classification
#)
# Training arguments


output_model = "csb-pol-25perc-oldpar-byte-sent"
training_args = TrainingArguments (
    output_dir="./byt5_small/"+output_model,
    overwrite_output_dir=True,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    evaluation_strategy = "epoch",
    #predict_with_generate=True,
   # save_steps=10_000,
    save_total_limit=3,
)

# Trainer
#trainer = Seq2SeqTrainer(
#    model=model,
#    args=training_args,
#    train_dataset=tokenized_dataset["train"],
#    eval_dataset=tokenized_dataset["validation"],
#    data_collator=data_collator,
#    tokenizer=tokenizer,
#)

# Pretrain the model
trainer.train()
trainer.push_to_hub()

259 260
Column([[87, 104, 117, 259, 113, 199, 136, 35, 105, 108, 125, 124, 102, 125, 113, 198, 172, 106, 198, 181, 35, 122, 198, 174, 102, 107, 198, 181, 122, 100, 113, 108, 198, 183, 49, 1], [74, 117, 199, 136, 103, 114, 111, 35, 109, 104, 35, 198, 181, 101, 118, 100, 103, 125, 114, 113, 124, 35, 113, 100, 35, 106, 198, 182, 259, 108, 104, 119, 104, 122, 102, 125, 108, 49, 1], [198, 149, 113, 35, 122, 35, 118, 122, 198, 181, 109, 108, 112, 35, 102, 125, 259, 102, 107, 102, 104, 49, 1], [68, 111, 115, 104, 109, 118, 110, 198, 183, 35, 198, 181, 117, 114, 106, 104, 113, 104, 125, 100, 35, 118, 115, 117, 100, 122, 108, 200, 133, 100, 35, 117, 198, 174, 118, 125, 113, 259, 114, 103, 125, 198, 172, 200, 133, 114, 122, 108, 102, 107, 35, 74, 198, 182, 117, 49, 1], [80, 198, 259, 125, 108, 35, 122, 104, 35, 90, 104, 118, 119, 117, 125, 198, 172, 103, 113, 124, 35, 72, 198, 188, 117, 114, 115, 108, 104, 35, 198, 188, 113, 108, 122, 104, 117, 118, 124, 119, 104, 119, 47, 35, 109, 100, 102, 125

AttributeError: 'TrainingArguments' object has no attribute 'generation_config'

In [2]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments
from datasets import load_dataset
import math, random
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments
import torch
tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
# tokenized_dataset = dataset.map(preprocess_function, batched=True)
model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")

sentinal_token_1 = tokenizer("<extra_id_0>", return_tensors="pt").to(model.device).input_ids # skip eos value "1"
sentinal_token_2 = tokenizer("<extra_id_1>", return_tensors="pt").to(model.device).input_ids # skip eos value "1"
extra_id_0 = int(sentinal_token_1[0][0])
extra_id_1 = int(sentinal_token_2[0][0])
print(extra_id_0, extra_id_1)

model_inputs = tokenizer("我是一条狗.")
#model_inputs = tokenizer(".")
input_ids = model_inputs.input_ids
attention_mask = model_inputs.attention_mask
print(input_ids, attention_mask, len(input_ids),len(attention_mask))

# mask 15% of the bytes in the tokenized sentence
length = round(len(input_ids) * 0.15) # mask 15% of character (T5 and mT5 do that for tokens)
max_start_index = len(input_ids) - length  # 2 chars -> 2-1=1 -> index 0 or 1 possible
random.seed(10)
start_index = random.randrange(1, max_start_index ) # leave out first and last character
exclusive_end_index = start_index + length
masked_input_ids = (  input_ids[:start_index] + [extra_id_0] + input_ids[exclusive_end_index :]  )
masked_labels_ids = ([extra_id_0]+ input_ids[start_index: exclusive_end_index] +[extra_id_1])

# update attention mask and labels
attention_mask = [1] * len(masked_input_ids)
model_inputs["attention_mask"] = attention_mask
model_inputs["labels"] = masked_labels_ids

print(input_ids, masked_input_ids,masked_labels_ids)
print(model_inputs)

ModuleNotFoundError: No module named 'datasets'

In [1]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments
from datasets import load_dataset

# Load domain-specific text (replace with your own corpus)
lang = "csb-pol-sent"
train_path = f"/dss/dsshome1/03/ge87wod2/morphological-inflection/preprocessing/preprocessing_to_json_or_tsv/data/{lang}_trn.tsv"
eval_path = f"/dss/dsshome1/03/ge87wod2/morphological-inflection/preprocessing/preprocessing_to_json_or_tsv/data/{lang}_dev.tsv"
dataset = load_dataset("csv", delimiter="\t",column_names=["input","labels"],data_files={"train": train_path,"validation": eval_path},quoting=3) # disable quote parsing


# Tokenizing
def preprocess_function(examples):
    inputs = examples["input"]
    model_inputs = tokenizer(inputs)
    labels = tokenizer(text_target=examples["labels"],padding=True) # same length irrelevant here, so no truncation or padding
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs




tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
tokenized_dataset = dataset.map(preprocess_function, batched=True)




# Load pre-trained model
model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, return_tensors="pt")
# Training arguments
training_args = TrainingArguments (
    output_dir="./byt5_small/"+lang+"-8epochs",
    overwrite_output_dir=True,
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy = "epoch",
    #predict_with_generate=True,
   # save_steps=10_000,
    save_total_limit=3,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# Pretrain the model
trainer.train()
trainer.push_to_hub()

/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-04-12 13:26:29.946699: I tensorflow/core/platform/cpu_feature_guard.cc:211] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# Fine-tuning small pre-trained ByT5 model using dataset

In [6]:
LANGS = ["grc","dan","fra","sme","deu","nav","jap","klr","eng","mul","deu_eng"]
LANGS = LANGS[-2:]
LANGS = ["mul_2","amh","pol","ote"]
LANGS = ["mul_2","amh","pol","ote"]
LANGS = ["mul-language-code","mul-language-word"]
LANGS = ["mul-language-code"]
LANGS = ["pol"]
#LANGS = ["slavic-family"]
#LANGS = ["slk"] # ces, slk, dsb
CHECKPOINT = "livles/csb-pol-byte-25perc-sent"
#CHECKPOINT = "google/byt5-small"


PATH_JSON_DATA = "../preprocessing/preprocessing_to_json_or_tsv/data/"

for lang in LANGS:
    from datasets import load_dataset
    dataset = load_dataset("json", data_files={"train": PATH_JSON_DATA + lang + "_trn.json", "validation": PATH_JSON_DATA + lang.split("-")[0] + "_dev.json"})

    print(dataset["train"][0])

    from transformers import AutoTokenizer

    checkpoint = "google/byt5-small"
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    # -

    import torch

    from transformers import (
        AutoTokenizer,
        TrainingArguments,
        Trainer,
        AutoModelForSeq2SeqLM,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer
    )
    from datasets import load_dataset

    def preprocess_function(examples):
        # inputs = [prefix + doc for doc in examples["text"]]
        inputs = examples["input"]
        model_inputs = tokenizer(inputs)

        labels = tokenizer(text_target=examples["target"]) # same length irrelevant here, so no truncation or padding

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


    tokenized_dataset = dataset.map(preprocess_function, batched=True)
    print(tokenized_dataset)
    sample = tokenized_dataset["train"][0]
    print(sample["labels"])

    from transformers import DataCollatorForSeq2Seq

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

    from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

    checkpoint = CHECKPOINT
    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    training_args = Seq2SeqTrainingArguments(
        output_dir="byt5_small/csb-pol-byte-25perc-sent-"+lang,
        evaluation_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=4,
        predict_with_generate=True,
        # save_strategy="epoch",
        fp16=False,# did not work with True #change to bf16=True for XPU 
        #bf16 = True, # performance was best without setting fp16 or bf16 to true
        push_to_hub=True,
        warmup_steps = 500,
        # load_best_model_at_end=True, # otherwise not model with minimum loss during training, like https://huggingface.co/docs/transformers/tasks/sequence_classification
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,   
    )

    trainer.train()
    trainer.push_to_hub()


{'input': 'Inflect zacny | ADJ;ACC(SG,FEM)', 'target': 'zacną'}


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['input', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['input', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})
[125, 100, 102, 113, 199, 136, 1]


config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,3.039700,0.276155
2,0.359800,0.112349
3,0.220500,0.077301
4,0.129800,0.071193


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

# Fine-Tuning with more epochs

In [8]:
!rm -rf /dss/dsshome1/03/ge87wod2/morphological-inflection/ByT5/byt5_small/*
LANGS = ["5lang-sample"]
#LANGS = ["slavic-family"]

PATH_JSON_DATA = "../preprocessing/preprocessing_to_json/data/"

for lang in LANGS:
    from datasets import load_dataset
    dataset = load_dataset("json", data_files={"train": PATH_JSON_DATA + lang + "_trn.json", "validation": PATH_JSON_DATA + lang + "_dev.json"})

    print(dataset["train"][0])

    from transformers import AutoTokenizer

    checkpoint = "google/byt5-small"
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    # 

    import torch

    from transformers import (
        AutoTokenizer,
        TrainingArguments,
        Trainer,
        AutoModelForSeq2SeqLM,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer
    )
    from datasets import load_dataset

    def preprocess_function(examples):
        # inputs = [prefix + doc for doc in examples["text"]]
        inputs = examples["input"]
        model_inputs = tokenizer(inputs)

        labels = tokenizer(text_target=examples["target"]) # same length irrelevant here, so no truncation or padding

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


    tokenized_dataset = dataset.map(preprocess_function, batched=True)
    print(tokenized_dataset)
    sample = tokenized_dataset["train"][0]
    print(sample["labels"])

    from transformers import DataCollatorForSeq2Seq

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

    from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    training_args = Seq2SeqTrainingArguments(
        output_dir="byt5_small/"+lang+"_13epochs",
        eval_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=13,
        predict_with_generate=True,
        fp16=False,# did not work with True #change to bf16=True for XPU 
        #bf16 = True, # performance was best without setting fp16 or bf16 to true
        push_to_hub=True,
        warmup_steps = 500,
        save_strategy="epoch",
        load_best_model_at_end=True, # otherwise not model with minimum loss during training, like https://huggingface.co/docs/transformers/tasks/sequence_classification,
        greater_is_better=False,
        metric_for_best_model="eval_loss"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        # compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.push_to_hub()


{'input': 'Navajo: Inflect náʼáłkad | V;IND;PFV;NOM(1,GRPL)', 'target': 'ńdaʼiilkad'}
DatasetDict({
    train: Dataset({
        features: ['input', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['input', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})
[200, 135, 103, 100, 205, 191, 108, 108, 111, 110, 100, 103, 1]


/tmp/ipykernel_3401446/2533773261.py:79: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
1,2.696000,0.714211
2,0.782000,0.506161
3,0.568900,0.431394
4,0.412900,0.378233
5,0.362600,0.339201
6,0.323500,0.316904
7,0.289100,0.290162
8,0.246900,0.279335


# Inference

In [8]:
# SETTINGS = ["grc","dan","fra","sme","deu","nav","jap","klr","eng","mul","deu_eng"] + ["mul_2","ote","pol","amh"] + ["mul-language-code","mul-language-word"]
SETTINGS = ["mul"]
SETTINGS = ["5lang-sample"]
BASE_TEST_LANGUAGES_OF_SETTING = {"slavic":["csb"],"slavic-word":["csb"],"slavic-family":["pol","ces","csb","dsb","slk"],"5lang-sample":["deu","nav","klr","amh","grc"],"5lang-sample_16epochs":["amh"],
                                 "sme-15ksamples":["sme"],"sme-20ksamples":["sme"],"klr-15ksamples":["klr"],"klr-20ksamples":["klr"],"slavic-distant-word": ["pol","ces","csb","dsb","slk","mkd","rus","bel"],"slavic-distant-word-or-family": ["pol","ces","csb","dsb","slk","mkd","rus","bel"],
                                  "csb-pol":["csb","pol"], "csb-pol-sent-pol": ["csb", "pol"], "csb-pol-sent-6epochs-pol": ["csb", "pol"],
                                  "csb-sent-6epochs-pol": ["csb", "pol"], "dsb":["dsb"],"slk":["slk"],"ces":["ces"],"grc":["grc"],"sme":["sme"],"klr":["klr"],"nav":["nav"],
                                  "mul-language-word":["klr"],"pol":["csb"],"amh":["amh"],"mul_2":["sme","klr","grc"],"csb-pol-byte-sent-pol":["csb"],"csb-pol-byte-20perc-sent-pol":["csb"],"csb-pol-byte-25perc-sent-pol":["csb"]
                                 }
CODES = {"slavic": [""],"slavic-word":["-lang-word"],"slavic-family":["-family","-lang-word","-word-or-family"],"5lang-sample":["-lang-word"],"5lang-sample_16epochs":["-lang-word"],"5lang-sample_13epochs":["-lang-word"],"sme-15ksamples":[""],"sme-20ksamples":[""],"klr-15ksamples":[""],"klr-20ksamples":[""],"slavic-distant-word":["-lang-word"],"slavic-distant-word-or-family":["-lang-word"],"csb-pol":[""],"csb-pol-sent-pol":[""],"csb-pol-sent-6epochs-pol": [""],"csb-sent-6epochs-pol": [""],"ces":[""],"slk":[""],"dsb":[""],"grc":[""],"sme":[""],"klr":[""],
        "mul-language-word":["-lang-word"],"nav":[""],"pol":[""],"amh":[""],"mul_2":[""],"csb-pol":[""],"csb-pol-byte-sent-pol":[""],"csb-pol-byte-20perc-sent-pol":[""],"csb-pol-byte-25perc-sent-pol":[""]}
SETTINGS = ["slavic","slavic-word"]
SETTINGS = ["sme-15ksamples","sme-20ksamples","klr-15ksamples","klr-20ksamples","5lang-sample_16epochs"]
SETTINGS = ["slavic-distant-word-or-family"]
SETTINGS = ["csb-sent-6epochs-pol"]
SETTINGS = ["slavic"]
SETTINGS = ["csb-pol-byte-25perc-sent-pol"]
#SETTINGS = ["mul-language-word"]
#TEST_LANGS_OLD = ["grc","dan","fra","sme","deu","nav","jap","klr","eng"] 
#TEST_LANGS = ["grc","dan","fra","sme","deu","nav","jap","klr","eng"] + ["ote","pol","amh","csb"]
#lang5_LANGS = ["deu","nav","klr","amh","grc"]
#lang_code = ""
ERROR_ANALYSIS_PATH = "/dss/dsshome1/03/ge87wod2/morphological-inflection/error_analysis_lucy/byt5"
for lang_setting in SETTINGS:
    test_langs = BASE_TEST_LANGUAGES_OF_SETTING[lang_setting]
    #if lang_setting == "mul": LANG_CODES = ["-language-word","-language-code",""] # test with no language code/word too
    #else: LANG_CODES = [""]
    #for lang_code in LANG_CODES:
        #if lang_setting in ( "deu_eng", "mul") and lang_code == "": test_langs = TEST_LANGS_OLD
        #elif lang_setting in ["mul_2","mul"]: test_langs = TEST_LANGS # test multilingual model on all languages it was trained on + Kashubian
        #elif lang_setting == "5lang-sample": test_langs = lang5_LANGS
        #elif lang_setting == "pol": test_langs = ["pol","csb"] # test Polish model on Kashubian also
        #else: 
    for lang_code in CODES[lang_setting]:
        for test_lang in test_langs:
            import torch
            from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
            PRETRAINED_MODEL = "google/byt5-small"
            FINETUNED_MODEL ="livles/"+lang_setting
            #FINETUNED_MODEL ="./byt5_small/"+lang_setting + lang_code
            MODEL = FINETUNED_MODEL
            print(MODEL)
            tokenizer = AutoTokenizer.from_pretrained(
                FINETUNED_MODEL,
            )
            model = AutoModelForSeq2SeqLM.from_pretrained(
                FINETUNED_MODEL,
            torch_dtype=torch.float16,
            device_map="auto"
            )
            from datasets import load_dataset
            OUT_DIR = "output_byt5_small/" + lang_setting + "/" 
            PATH_TO_JSON_TEST_FILE = "../preprocessing/preprocessing_to_json_or_tsv/data/" + test_lang + lang_code+ "_tst.json"
            #ERROR_ANALYSIS_PATH = "/dss/dsshome1/03/ge87wod2/morphological-inflection/error_analysis_lucy/byt5/" + lang_setting + "/" + test_lang + lang_code + ".errors"
            print("test file: ",PATH_TO_JSON_TEST_FILE)
            PATH_TO_ORIG_TEST_FILE = "/dss/dsshome1/03/ge87wod2/morphological-inflection/2023InflectionST/part1/data/" + test_lang + ".tst"
            from pathlib import Path
            Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
            with open (OUT_DIR + test_lang + lang_code + "_byt5-small.out","w") as out_file, open (PATH_TO_ORIG_TEST_FILE, "r") as test_file: #, open (ERROR_ANALYSIS_PATH, "w") as error_file
                dataset = load_dataset("json",data_files={"test":PATH_TO_JSON_TEST_FILE})
                covered_test_lines = dataset["test"]["input"]
                LEN = len(covered_test_lines)
                COUNTER = 0

                # preds = map (generate,covered_test_lines)
                for line in covered_test_lines:
                    input_ids = tokenizer(line, return_tensors="pt").to(model.device)
                    output = model.generate(
                        **input_ids,
                        max_new_tokens=50,
                        num_beams=4,
                        #early_stopping=True
                    )
                    output_string = (tokenizer.decode(output[0], skip_special_tokens=True))
                    ref_line = test_file.readline()
                    lemma, features, ref = ref_line.strip().split("\t")

                    # evaluate
                    if (ref == output_string):
                        COUNTER += 1
                        print(ref,output_string)
                    else:
                        print("ref:",ref,"pred:",output_string)
                     #   error_file.write("ref\t"+ref_line+"\n")
                      #  error_file.write("byt5\t"+line+"\n\n")
                       # error_file.flush()
                    # write
                    out_file.write(lemma + "\t" +features+ "\t" + output_string + "\n")
                    out_file.flush()

                accuracy = COUNTER / LEN
                out_file.write("accuracy:"+ str(accuracy)+"\n")
                print(accuracy)

livles/csb-pol-byte-25perc-sent-pol


tokenizer_config.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

test file:  ../preprocessing/preprocessing_to_json_or_tsv/data/csb_tst.json
ref: narzãdzacze pred: narzãdzôcze
ref: narzãdzaczom pred: narzãdzôczom
ref: narzãdzaczu pred: narzãdzôczu
ref: narzãdzaczama pred: narzãdzôczami
ref: narzãdzaczu pred: narzãdzôczu
ref: narzãdzaczowi pred: narzãdzôczowi
ref: narzãdzacza pred: narzãdzôcza
ref: narzãdzaczach pred: narzãdzôczach
ref: narzãdzaczów pred: narzãdzôczy
ref: narzãdzaczã pred: narzãdzôczem
ref: narzãdzacze pred: narzãdzôcze
ref: narzãdzacza pred: narzãdzôcz
narzãdzôcz narzãdzôcz
ref: narzãdzacze pred: narzãdzôcze
Jastrë Jastrë
Jastrach Jastrach
ref: Jastróm pred: Jastrom
Jastrów Jastrów
ref: Jastrama pred: Jastrë
Jastrë Jastrë
Jastrë Jastrë
ref: pòniedzałczi pred: pòniedzôłki
ref: pòniedzałkóm pred: pòniedzôłkom
ref: pòniedzałkù pred: pòniedzôłku
ref: pòniedzałkama pred: pòniedzôłkami
ref: pòniedzałkù pred: pòniedzôłcu
ref: poniedzałkòwi pred: pòniedzôłcowi
ref: pòniedzałkù pred: pòniedzôłku
ref: pòniedzałkach pred: pòniedzôłkach
ref: pò

In [32]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset
import random
import numpy
# get tokenizer and sentinal tokens for masking
tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")

# Load pre-trained model
model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, return_tensors="pt")
random.seed(10)

def preprocess_function(examples): # receives unmasked sentence as input
    inputs = examples
    inputs = inputs.strip()
    model_inputs = tokenizer(inputs)
    input_ids = model_inputs.input_ids
    attention_mask = model_inputs.attention_mask
    # print(input_ids, attention_mask, len(input_ids),len(attention_mask))
    # mask 15% of the bytes randomly in the tokenized sentence leaving out bordering bytes.
    len_sentence = len(input_ids)
    length = round(len(input_ids) * 0.22) # mask 15% of character (T5 and mT5 do that for tokens)
    return length



from array import array
# load dataset
lang = "csb-pol-sent"
train_path = f"/dss/dsshome1/03/ge87wod2/morphological-inflection/preprocessing/preprocessing_to_json_or_tsv/data/{lang}_trn.tsv"
eval_path = f"/dss/dsshome1/03/ge87wod2/morphological-inflection/preprocessing/preprocessing_to_json_or_tsv/data/{lang}_dev.tsv"
dataset = load_dataset("csv", delimiter="\t",column_names=["input"],data_files={"train": train_path,"validation": eval_path},quoting=3) # disable quote parsing
print(dataset["train"][1])
mean_train = sum(list(map(preprocess_function, dataset["train"]["input"]))) / 9000
mean_dev = sum( list( map(preprocess_function, dataset["validation"]["input"]) ) ) / 1000

print("mean" ,mean_train,mean_dev)


{'input': 'Grądol je òbsadzony na górze drobné wietewczi.'}
mean 21.695666666666668 21.667
